# Session 13 — 2/3: reversible arms (euler vs midpoint)

Same model, same data, same seed, **same batch size as the baseline**. The only
change is the inter-layer update rule and the fact that activations are not
stored at all.

`--check_recon` logs the live reconstruction error during training: how far the
rebuilt activation drifts from the real one. That is the number that decides
which variant is trustworthy, not just which is fast.

In [ ]:
!nvidia-smi
!git clone https://github.com/rjvim/era-v5-session13-reversibility repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, 'src')

In [ ]:
import json
BATCH_FIX = json.load(open('results/maxbatch_baseline.json'))['max_batch']
print('fixed batch =', BATCH_FIX)

## Run 2a — euler

In [ ]:
!python src/train.py --mode euler --batch_size {BATCH_FIX} --run_name euler_fixed     --h 0.5 --euler_iters 8 --check_recon --seq_len 512 --total_tokens 50000000 --resume


## Run 2b — midpoint

In [ ]:
!python src/train.py --mode midpoint --batch_size {BATCH_FIX} --run_name midpoint_fixed     --h 0.5 --check_recon --seq_len 512 --total_tokens 50000000 --resume


## Compare

In [ ]:
import json, matplotlib.pyplot as plt
runs = {k: json.load(open(f'results/{k}.json'))
        for k in ['baseline_fixed', 'euler_fixed', 'midpoint_fixed']}
for k, d in runs.items():
    print(f"{k:16s} loss {d['final_train_loss']:.4f} val {d['final_val_loss']:.4f} "
          f"{d['tok_per_s']:>9,.0f} tok/s  peak {d['peak_mem_gb']:.2f} GB  "
          f"recon {d['max_recon_err'] if d['max_recon_err'] else 0:.2e}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, d in runs.items():
    ax[0].plot([r['step'] for r in d['log']], [r['loss'] for r in d['log']], label=k)
    if d['recon_err_log']:
        ax[1].semilogy([r[0] for r in d['recon_err_log']],
                       [r[1] for r in d['recon_err_log']], label=k)
ax[0].set(xlabel='step', ylabel='train loss', title='Loss trajectory'); ax[0].legend()
ax[1].set(xlabel='step', ylabel='relative recon error', title='Activation reconstruction error')
ax[1].legend(); plt.tight_layout(); plt.savefig('results/curves.png', dpi=120); plt.show()

Pick the winner on **loss trajectory first, reconstruction error second, speed third** — a fast variant with a diverging inverse is training on wrong gradients.